In [1]:
# 03_fds_fairness_auto.ipynb
# Insurance fraud detection fairness (Auto Claims): false-positive disparity and
# false-positive / restoration metrics versus accuracy-only tuning (T-4, T-7, T-9).
# All code is contained in this single cell.

import warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, confusion_matrix
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

def find_root():
    p = Path.cwd().resolve()
    for c in [p, *p.parents]:
        if (c / "results").is_dir() and (c / "notebooks").is_dir():
            return c
    return p.parent if p.name == "notebooks" else p
ROOT = find_root()
DATA, FIG, TAB = ROOT / "data", ROOT / "results" / "figures", ROOT / "results" / "tables"
for d in (DATA, FIG, TAB):
    d.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 600, "font.size": 11,
                     "axes.edgecolor": "0.2", "grid.color": "0.85"})
G_DARK, G_LIGHT = "0.25", "0.65"
def savefig(fig, stem):
    for e in ("png", "pdf"):
        fig.savefig(FIG / (stem + "." + e), dpi=600, bbox_inches="tight")
    plt.close(fig)
def savetable(df, stem, index=False):
    df.to_csv(TAB / (stem + ".csv"), index=index)
    try:
        df.to_latex(TAB / (stem + ".tex"), index=index, float_format="%.3f")
    except Exception:
        pass

fp = DATA / "fraud_autoclaims.csv"
if not fp.exists():
    url = "https://raw.githubusercontent.com/mwitiderrick/insurancedata/master/insurance_claims.csv"
    pd.read_csv(url).to_csv(fp, index=False)
df = pd.read_csv(fp).replace("?", np.nan)
y = (df["fraud_reported"].astype(str).str.upper() == "Y").astype(int)
sex = df["insured_sex"].astype(str).str.upper()

# drop identifiers / free-text / leakage / target
drop = ["fraud_reported", "policy_number", "policy_bind_date", "insured_zip",
        "incident_date", "incident_location", "_c39", "auto_model"]
X = df.drop(columns=[c for c in drop if c in df.columns])
num_cols = X.select_dtypes("number").columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
X[cat_cols] = X[cat_cols].astype(str).fillna("missing")

pre = ColumnTransformer([("num", StandardScaler(), num_cols),
                         ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)])
clf = Pipeline([("pre", pre), ("gb", GradientBoostingClassifier(random_state=SEED))])
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
proba = cross_val_predict(clf, X, y, cv=cv, method="predict_proba")[:, 1]
auc = roc_auc_score(y, proba)

# ---- accuracy-optimal threshold, then group false-positive rates ----
ths = np.linspace(0.05, 0.95, 91)
acc = [accuracy_score(y, (proba >= t).astype(int)) for t in ths]
t_acc = float(ths[int(np.argmax(acc))])
pred = (proba >= t_acc).astype(int)

rows = []
for g in ["MALE", "FEMALE"]:
    mask = (sex == g).values
    yy = y[mask].values
    pp = pred[mask]
    tn, fp, fn, tp = confusion_matrix(yy, pp, labels=[0, 1]).ravel()
    rows.append({"insured_sex": g, "n": int(mask.sum()),
                 "false_positive_rate": fp / (fp + tn) if (fp + tn) else np.nan,
                 "recall_tpr": tp / (tp + fn) if (tp + fn) else np.nan,
                 "flag_rate": float(pp.mean())})
grp = pd.DataFrame(rows)
grp["fpr_gap"] = grp["false_positive_rate"].max() - grp["false_positive_rate"].min()
savetable(grp, "fds_group_metrics")

# ---- restoration: a human-review gate on lower-confidence flags recovers FPs (T-7) ----
flagged = pred == 1
band_cut = np.quantile(proba[flagged], 0.5)
review_band = flagged & (proba < band_cut)
fp_mask = flagged & (y.values == 0)
fp_total = int(fp_mask.sum())
fp_restored = int((fp_mask & review_band).sum())
restoration = pd.DataFrame([{"accuracy_threshold": round(t_acc, 3),
    "false_positives_total": fp_total, "fp_sent_to_review": int(review_band.sum()),
    "fp_restored_by_review": fp_restored,
    "restoration_rate": fp_restored / fp_total if fp_total else np.nan}])
savetable(restoration, "fds_restoration")

# ---- threshold tradeoff curve ----
tpr_list = [float(((proba >= t).astype(int)[y.values == 1]).mean()) for t in ths]
fpr_list = [float(((proba >= t).astype(int)[y.values == 0]).mean()) for t in ths]
savetable(pd.DataFrame({"threshold": ths, "tpr": tpr_list, "fpr": fpr_list}),
          "fds_threshold_tradeoff")

# ---- figures ----
fig, ax = plt.subplots(figsize=(4.2, 3.2))
sns.barplot(data=grp, x="insured_sex", y="false_positive_rate", ax=ax,
            palette=[G_DARK, G_LIGHT], edgecolor="black")
ax.set_xlabel(""); ax.set_ylabel("False positive rate")
savefig(fig, "fig_fds_fpr_by_sex")

fpr, tpr, _ = roc_curve(y, proba)
fig, ax = plt.subplots(figsize=(4.2, 3.4))
ax.plot(fpr, tpr, color=G_DARK, lw=1.8)
ax.plot([0, 1], [0, 1], color=G_LIGHT, ls="--", lw=1.2)
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.text(0.52, 0.10, "AUC = %.3f" % auc)
savefig(fig, "fig_fds_roc")

fig, ax = plt.subplots(figsize=(4.6, 3.4))
ax.plot(ths, tpr_list, color=G_DARK, lw=1.8, label="TPR (fraud caught)")
ax.plot(ths, fpr_list, color=G_LIGHT, lw=1.8, ls="--", label="FPR (false accusations)")
ax.axvline(t_acc, color="0.1", lw=1.0, ls=":")
ax.set_xlabel("Decision threshold"); ax.set_ylabel("Rate")
ax.legend(frameon=False, fontsize=9)
savefig(fig, "fig_fds_threshold_tradeoff")

print("fds leg done | AUC: %.3f" % auc, "| acc-threshold: %.2f" % t_acc,
      "| FPR gap(sex): %.3f" % grp["fpr_gap"].iloc[0],
      "| restoration rate: %.3f" % restoration["restoration_rate"].iloc[0])


fds leg done | AUC: 0.852 | acc-threshold: 0.18 | FPR gap(sex): 0.009 | restoration rate: 0.476
